In [1]:
# 1. 安裝必要套件 (如果 Colab 尚未安裝)
!pip install gradio faiss-cpu torch torchvision pillow numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00


In [2]:
import os
import glob
import numpy as np
import faiss
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
import gradio as gr

# ==============================================================================
# 設定與參數
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用裝置: {DEVICE}")

# 圖像預處理 (兩個模型共用)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

使用裝置: cuda


In [ ]:
ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
# 解壓縮到 Colab 的根目錄
!unzip -q {ZIP_PATH} -d /content/

# 資料庫圖片路徑
DATABASE_IMAGES_DIR = "/content/train/images"  # 請確認此路徑正確

In [ ]:

import sys


# ==============================================================================
# 模型 1: Baseline (ResNet50) 載入邏輯
# ==============================================================================
def load_baseline_model(model_path):
    print("正在載入 Baseline 模型...")
    model = resnet50(weights=None)
    model.fc = nn.Identity()  # 移除最後一層，輸出 2048 維

    if os.path.exists(model_path):
        try:
            state_dict = torch.load(model_path, map_location=DEVICE)
            # 處理可能多出的 fc 層權重
            if 'fc.weight' in state_dict:
                 state_dict.pop('fc.weight')
                 state_dict.pop('fc.bias')
            model.load_state_dict(state_dict, strict=False)
        except Exception as e:
            print(f"Baseline 權重載入警告: {e}")
    else:
        print(f"找不到 Baseline 權重檔: {model_path}，使用隨機權重(僅供測試UI)")

    model.to(DEVICE)
    model.eval()
    return model

# ==============================================================================
# 模型 2: Advanced (EmbeddingNet) 載入邏輯
# ==============================================================================
def load_advanced_model(model_path, embedding_size=512):
    print("正在載入 Advanced 模型...")
    # 嘗試導入 model.py
    try:
        from mode1 import EmbeddingNet
        try:
            model = EmbeddingNet(emb_size=embedding_size)
        except TypeError as e:
            if "unexpected keyword argument 'emb_size'" in str(e):
                print(f"⚠️ 警告: 'mode1.py' 中的 EmbeddingNet.__init__ 不接受 'emb_size' 參數。")
                print("將嘗試不帶 'emb_size' 參數初始化模型，請檢查您的 mode1.py 文件。")
                model = EmbeddingNet() # Try without emb_size
            else:
                raise e # Re-raise other TypeErrors
    except ImportError:
        print("⚠️ 警告: 找不到 'model.py'。使用臨時定義的 Dummy Model 替代 (僅供測試UI)。")
        # 這裡定義一個臨時的結構以防報錯，實際使用請務必上傳 model.py
        class EmbeddingNet(nn.Module):
            def __init__(self, emb_size):
                super().__init__()
                self.backbone = resnet50(weights=None)
                self.backbone.fc = nn.Linear(2048, emb_size)
            def forward(self, x): return self.backbone(x)
        model = EmbeddingNet(emb_size=embedding_size)

    if os.path.exists(model_path):
        try:
            state_dict = torch.load(model_path, map_location=DEVICE)

            # --- 🛠️ 新增：處理可能的 state_dict 嵌套 ---
            if 'model_state' in state_dict:
                state_dict = state_dict['model_state']
            elif 'state_dict' in state_dict:
                state_dict = state_dict['state_dict']

            # --- 🛠️ 新增：Debug 打印 Key 的前綴 ---
            print("\n🔍 --- 權重載入診斷 ---")
            ckpt_keys = list(state_dict.keys())
            model_keys = list(model.state_dict().keys())
            print(f"檔案中的 Keys (前3個): {ckpt_keys[:3]}")
            print(f"模型中的 Keys (前3個): {model_keys[:3]}")

            # 常見問題：DataParallel 造成的 'module.' 前綴
            if ckpt_keys[0].startswith('module.') and not model_keys[0].startswith('module.'):
                print("⚠️ 偵測到 'module.' 前綴，正在修正...")
                new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
                state_dict = new_state_dict

            # --- 嘗試載入 ---
            # 把 strict 改成 True 來看看真正的報錯是什麼！
            model.load_state_dict(state_dict, strict=True)
            print("✅ Advanced 模型權重載入成功！")

        except RuntimeError as e:
            print(f"❌ 權重形狀不匹配 (請檢查 strict=False 的後果): {e}")
            print("嘗試使用 strict=False 繼續 (但這可能是導致輸出固定的原因)...")
            model.load_state_dict(state_dict, strict=False)
        except Exception as e:
            print(f"Advanced 權重載入未知錯誤: {e}")
    else:
        print(f"❌ 找不到檔案: {model_path}")

    model.to(DEVICE)
    model.eval()
    return model

# ==============================================================================
# 系統初始化 (載入模型與索引)
# ==============================================================================

# 1. 載入模型
baseline_model = load_baseline_model("/content/drive/MyDrive/baseline_model.pth")
advanced_model = load_advanced_model("/content/drive/MyDrive/advanced_best_model2.pth", embedding_size=512)

# 2. 載入 FAISS 索引與檔名列表
def load_db(index_path, names_path):
    if not os.path.exists(index_path) or not os.path.exists(names_path):
        print(f"❌ 檔案缺失: {index_path} 或 {names_path}")
        return None, []

    try:
        index = faiss.read_index(index_path)
        if names_path.endswith('.npz'):
            names = np.load(names_path, allow_pickle=True)['filenames']
        else:
            names = np.load(names_path, allow_pickle=True)
        return index, names
    except Exception as e:
        print(f"載入資料庫失敗: {e}")
        return None, []

# 載入 Baseline 資料庫
index_base, names_base = load_db("/content/drive/MyDrive/faiss_deepfashion_retrieval.index", "/content/drive/MyDrive/indexed_file_names.npy")

# 載入 Advanced 資料庫
index_adv, names_adv = load_db("/content/drive/MyDrive/faiss_index_gallery.bin", "/content/drive/MyDrive/indexed_advanced.npz")

# ==============================================================================
# 核心檢索邏輯
# ==============================================================================
def retrieve(model, index, names, img_pil, k, normalize=False):
    if index is None or model is None:
        return []

    # 轉 Tensor
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    # 提取特徵
    with torch.no_grad():
        vector = model(img_tensor)

        # --- 🔴 DEBUG 區塊開始 ---
        # 觀察前 5 個數值，如果每次上傳不同照片，這串數字完全一樣，代表模型輸入有問題
        sample_vals = vector[0][:5].cpu().numpy()
        print(f"[{'Advanced' if normalize else 'Baseline'}] 特徵向量前5碼: {sample_vals}")
        # --- 🔴 DEBUG 區塊結束 ---

        if normalize:
            vector = F.normalize(vector, p=2, dim=1)
        vector = vector.cpu().numpy()

    # Flatten
    if len(vector.shape) > 2: vector = vector.reshape(1, -1)

    # FAISS 搜尋
    D, I = index.search(vector, k)

    # (後續抓取圖片的程式碼保持不變...)
    retrieved_images = []
    for idx_in_db in I[0]:
        if idx_in_db != -1 and idx_in_db < len(names):
            file_name = names[idx_in_db]
            if "/" in file_name or "\\" in file_name:
                file_name = os.path.basename(file_name)
            full_path = os.path.join(DATABASE_IMAGES_DIR, file_name)
            if os.path.exists(full_path):
                retrieved_images.append((Image.open(full_path), file_name))

    return retrieved_images

# ==============================================================================
# Gradio 介面邏輯
# ==============================================================================
def search_pipeline(input_image, top_k):
    if input_image is None:
        return [], []

    # 執行 Baseline 檢索 (不需額外 normalize，視訓練方式而定，這裡參照 Noteook1)
    results_base = retrieve(baseline_model, index_base, names_base, input_image, top_k, normalize=False)

    # 執行 Advanced 檢索 (Notebook2 中有 F.normalize)
    results_adv = retrieve(advanced_model, index_adv, names_adv, input_image, top_k, normalize=True)

    return results_base, results_adv

# 建立 UI
with gr.Blocks(title="DeepFashion 雙模型比較檢索 Demo", css="footer {visibility: hidden}") as demo:

    gr.Markdown(
        """
        # <center>👗 DeepFashion 雙模型比較檢索 Demo</center>
        請上傳一張服飾圖片以進行以圖搜圖。系統將同時比較 **Baseline (您的對照模型)** [上] 和 **改動版 (您的主模型)** [下] 的檢索結果。
        """
    )

    with gr.Row():
        # --- 左側控制區 ---
        with gr.Column(scale=1):
            input_img = gr.Image(label="【1. 查詢圖片】 請上傳服飾圖片", type="pil", height=300)

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=10,
                value=5,
                step=1,
                label="【2. 檢索數量】 請選擇 Top K"
            )

            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary")
                submit_btn = gr.Button("Submit", variant="primary")

        # --- 右側結果區 ---
        with gr.Column(scale=2):
            gr.Markdown("### 🔆 檢索結果 (上方: Baseline (您的對照模型))")
            gallery_base = gr.Gallery(
                label="Baseline Results",
                show_label=False,
                columns=[5],
                rows=[1],
                object_fit="contain",
                height=250
            )

            gr.Markdown("### 🔆 檢索結果 (下方: 改動版 (您的主模型))")
            gallery_adv = gr.Gallery(
                label="Advanced Results",
                show_label=False,
                columns=[5],
                rows=[1],
                object_fit="contain",
                height=250
            )

    # 綁定事件
    submit_btn.click(
        fn=search_pipeline,
        inputs=[input_img, top_k_slider],
        outputs=[gallery_base, gallery_adv]
    )

    # Clear 按鈕邏輯
    def clear_all():
        return None, [], []

    clear_btn.click(
        fn=clear_all,
        inputs=None,
        outputs=[input_img, gallery_base, gallery_adv]
    )

# 啟動應用
print("啟動 Gradio 介面...")
demo.queue().launch(share=True, debug=True)
